In [1]:
!pip install langgraph langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.5 MB/s eta 0:00:00


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key=" ",
    model="llama-3.1-8b-instant",
    temperature=0
)

In [3]:
from typing import TypedDict, List

class ChatState(TypedDict):
    messages: List[str]  # The running conversation history
    name: str            # Remembered user name
    preferences: str     # Remembered preferences

In [4]:
def chatbot(state: ChatState) -> dict:
    user_msg = state['messages'][-1]

    # Extract current memory or default to empty strings
    name = state.get('name', '')
    prefs = state.get('preferences', '')

    # Simple extraction logic based on keywords
    if 'my name is' in user_msg.lower():
        name = user_msg.lower().split('my name is')[-1].strip().title()

    if 'i like' in user_msg.lower():
        prefs = user_msg.lower().split('i like')[-1].strip()

    # Inject background context directly into the prompt frame
    context = f"The user's name is {name}. They like {prefs}."
    reply = llm.invoke(context + ' Reply to: ' + user_msg).content

    # Return updates back into the tracking state loop
    return {
        'messages': state['messages'] + [reply],
        'name': name,
        'preferences': prefs
    }

In [5]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(ChatState)
builder.add_node('chatbot', chatbot)
builder.add_edge(START, 'chatbot')
builder.add_edge('chatbot', END)

graph = builder.compile()

In [ ]:
state = {'messages': [], 'name': '', 'preferences': ''}

while True:
    user = input('You: ')
    if user.lower() == 'quit':
        break

    state['messages'].append(user)
    state = graph.invoke(state)
    print('Bot:', state['messages'][-1])

You: Hi my name is gnapika
Bot: Hi Gnapika, nice to meet you. What brings you here today?
You: I like cats
Bot: Me too, Gnapika. There's something so endearing about their little faces and playful personalities. Do you have a cat at home, or is there a particular breed that you're fond of?
You: i have 2 cats at home
Bot: That's great, Gnapika. Cats are such wonderful companions. Do you have any funny stories about your two feline friends?
